In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple, List, Dict, Optional

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Sklearn
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# GC-IMS loader library
import ims  # gc-ims-tools
from ims import Spectrum

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
class Config:
    """
    Central configuration for the GC-IMS augmentation pipeline.
    Dimensions (M, N) will be inferred from the first .mea spectrum
    after binning & optional cutting.
    """

    # Will be determined dynamically after loading the first spectrum
    M: Optional[int] = None   # Retention time dimension
    N: Optional[int] = None   # Drift time dimension

    # Autoencoder parameters
    D = 32  # Latent dimension

    # Transformer architecture
    NHEAD = 4
    NUM_LAYERS = 2
    DIM_FEEDFORWARD = 128
    DROPOUT = 0.1

    # Training parameters
    BATCH_SIZE = 64
    LEARNING_RATE = 5e-4
    NUM_EPOCHS = 150
    PATIENCE = 15
    TRAIN_SPLIT = 0.85

    # Undersampling flat signals
    UNDERSAMPLE_PROB = 0.25

    # Paths
    DATA_PATH = "data/"          # Folder containing class subfolders with .mea files
    MODEL_PATH = "models/"       # Where autoencoder checkpoints are stored
    RESULTS_PATH = "results/"    # Where plots & results will be saved

config = Config()

# Create directories if needed
Path(config.MODEL_PATH).mkdir(exist_ok=True)
Path(config.RESULTS_PATH).mkdir(exist_ok=True)

print("Config initialized.")


In [ ]:
def pool_rt_axis(X: np.ndarray, target_M: int = 512) -> np.ndarray:
    """
    Downsample the retention-time axis by average pooling.

    X shape: (n_samples, M, N)
    target_M: approximate target number of RT bins (sequence length for AE1)

    Returns:
        X_pooled : (n_samples, M_new, N)
    """
    n_samples, M, N = X.shape

    if M <= target_M:
        print(f"No RT pooling applied (M={M} <= target_M={target_M}).")
        return X

    # Compute integer pooling factor
    factor = int(np.floor(M / target_M))
    M_new = M // factor

    # Trim to a multiple of factor
    M_trim = M_new * factor
    if M_trim != M:
        print(f"Trimming RT from {M} → {M_trim} to be divisible by factor={factor}.")
        X = X[:, :M_trim, :]

    # Reshape and average-pool along RT axis
    X_reshaped = X.reshape(n_samples, M_new, factor, N)
    X_pooled = X_reshaped.mean(axis=2)

    print(f"RT pooling: M={M} → M_new={M_new} with factor={factor}")

    # Update global config.M for AE1
    config.M = M_new
    print(f"Config.M updated to {config.M}")

    return X_pooled


In [ ]:
def get_mea_files_and_labels(root_path: str) -> Tuple[List[Path], np.ndarray]:
    """
    Recursively collect all .mea files under the dataset directory.
    Returns:
        files  - list of full paths to .mea files
        labels - numpy array of string labels (folder names)
    """
    root = Path(root_path)

    files: List[Path] = []
    labels: List[str] = []

    if not root.exists():
        raise RuntimeError(f"DATA_PATH does not exist: {root}")

    for class_dir in sorted(root.iterdir()):
        if class_dir.is_dir():
            label = class_dir.name
            mea_files = sorted(class_dir.rglob("*.mea"))

            for f in mea_files:
                files.append(f)
                labels.append(label)

    if len(files) == 0:
        raise RuntimeError(
            f"No .mea files found under {root_path}. "
            f"Make sure folder structure is: data/<class_name>/*.mea"
        )
    return files, np.array(labels)


In [ ]:
def spectrum_to_matrix(
    spec: "ims.Spectrum",
    *,
    bin_factor: int = 2,
    cut_rt: Optional[Tuple[float, float]] = None,
    cut_dt: Optional[Tuple[float, float]] = None,
    normalize: bool = False
) -> np.ndarray:

    # Work on a copy so the original spectrum is not altered
    s = spec.copy()

    # --- Optional retention time cutting ---
    if cut_rt is not None:
        start, stop = cut_rt
        s.cut_rt(start, stop)

    # --- Optional drift time cutting ---
    if cut_dt is not None:
        start, stop = cut_dt
        s.cut_dt(start, stop)

    # Get intensity matrix BEFORE binning
    mat = s.values.astype(np.float32)

    # --- Numpy-based SAFE binning ---
    if bin_factor is not None and bin_factor > 1:
        rt = mat.shape[0] - (mat.shape[0] % bin_factor)
        dt = mat.shape[1] - (mat.shape[1] % bin_factor)

        mat = mat[:rt, :dt]   # crop to divisible size

        mat = mat.reshape(
            rt // bin_factor, bin_factor,
            dt // bin_factor, bin_factor
        ).mean(axis=(1, 3))

    if normalize:
        mat = mat / (mat.max() + 1e-10)

    return mat


In [ ]:
def load_spectral_data(
    data_path: str,
    *,
    bin_factor: int = 2,                      # Your chosen compression
    cut_rt: Optional[Tuple[float, float]] = None,
    cut_dt: Optional[Tuple[float, float]] = None,
    normalize: bool = False
) -> Tuple[np.ndarray, np.ndarray]:

    # Step 1: Find .mea files

    files, labels = get_mea_files_and_labels(data_path)
    n_samples = len(files)
    print(f"\nFound {n_samples} spectra.")

    # Step 2: Load & preview first spectrum using ims

    first_path = files[0]
    print(f"\nLoading first spectrum for preview: {first_path}")

    first_spec = ims.Spectrum.read_mea(str(first_path))
    print("Original shape:", first_spec.values.shape)


    # Step 3: Convert first spectrum → matrix

    mat0 = spectrum_to_matrix(
        first_spec,
        bin_factor=bin_factor,
        cut_rt=cut_rt,
        cut_dt=cut_dt,
        normalize=normalize
    )

    print("Shape:", mat0.shape)
    print("Min:", mat0.min(), "Max:", mat0.max(), "Mean:", mat0.mean())
    print("Median:", np.median(mat0))
    print("Percentiles:", np.percentile(mat0, [0, 1, 5, 50, 95, 99, 100]))

    unique_vals = np.unique(mat0)
    print("Unique values:", len(unique_vals))
    print("First 20 unique:", unique_vals[:20])


    M, N = mat0.shape
    print(f"Compressed shape (after ims & binning={bin_factor}): {mat0.shape}")

    # Update global config
    config.M = M
    config.N = N
    print(f"Config updated: M={config.M}, N={config.N}")

    # Step 4: Allocate X array

    X = np.empty((n_samples, M, N), dtype=np.float32)
    X[0] = mat0

    # Step 5: Load remaining spectra
    for i, path in enumerate(files[1:], start=1):
        print(f"[{i+1}/{n_samples}] Loading {path}")

        spec = ims.Spectrum.read_mea(str(path))
        mat = spectrum_to_matrix(
            spec,
            bin_factor=bin_factor,
            cut_rt=cut_rt,
            cut_dt=cut_dt,
            normalize=normalize
        )

        # Safety check
        if mat.shape != (M, N):
            raise ValueError(
                f"Shape mismatch in {path}: expected {(M, N)}, got {mat.shape}"
            )

        X[i] = mat

    print("\nFinished loading dataset.")
    print("Final X shape:", X.shape)
    print("Unique labels:", np.unique(labels))

    return X, labels


In [ ]:
X_raw, y_raw = load_spectral_data(
    config.DATA_PATH,
    bin_factor=2,         # compression
    cut_rt=None,          # Optionally set (start, stop)
    cut_dt=None,          # Optionally set (start, stop)
    normalize=False       # ims normalization
)

print("\nLoaded dataset:")
print("X_raw shape =", X_raw.shape)
print("Labels =", sorted(set(y_raw)))
print("Config.M, Config.N =", config.M, config.N)

mat = X_raw[0]

# Remove negative values
mat = np.maximum(mat, 0)

# RIP clipping (helps visualize real peaks)
hi = np.percentile(mat, 99.9)
mat_adj = np.clip(mat, 0, hi)


In [ ]:
class SpectralDataPreprocessor:
    """
    Steps:
      1) Clip negative intensities
      2) Log10(1 + intensity)
      3) Global min–max scaling to [0, 1]
    """

    def __init__(self):
        self.log_min = None
        self.log_max = None

    def log_scale_normalize(self, data: np.ndarray, fit: bool = False) -> np.ndarray:
        eps = 1e-10

        # 1) Zero clip negative values
        data = np.maximum(data, 0)

        # 2) Log10 transform (paper uses log base 10)
        log_data = np.log10(data + 1.0)

        # Fit global min/max once
        if fit:
            self.log_min = np.min(log_data)
            self.log_max = np.max(log_data)

        # 3) Normalize
        norm = (log_data - self.log_min) / (self.log_max - self.log_min + eps)
        return norm.astype(np.float32)

    def inverse_transform(self, norm_data: np.ndarray) -> np.ndarray:
        eps = 1e-10

        log_data = norm_data * (self.log_max - self.log_min) + self.log_min
        data = (10**log_data) - 1.0

        # Avoid negative reconstruction due to numerical noise
        data = np.maximum(data, 0)
        return data.astype(np.float32)

    def preprocess_dataset(self, X: np.ndarray, y: np.ndarray, *, fit: bool = False):
        Xp = np.empty_like(X, dtype=np.float32)
        for i in range(len(X)):
            Xp[i] = self.log_scale_normalize(X[i], fit=fit)
        return Xp, y



In [ ]:
# Convert string labels to integer classes
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)

print("Classes:", list(label_encoder.classes_))

# Initialize preprocessor
preprocessor = SpectralDataPreprocessor()

# Fit + transform on entire dataset
X_processed, y_processed = preprocessor.preprocess_dataset(
    X_raw,
    y_encoded,
    fit=True
)

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Standard sinusoidal positional encoding.
    Injects sequence index info into transformer inputs.
    """

    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            x + positional encoding
        """
        seq_len = x.size(1)
        return x + self.pe[:seq_len].transpose(0, 1)


In [ ]:
class TransformerAutoencoder(nn.Module):
    """
    Transformer autoencoder for 1D timeseries.

    Encoder:
        Input (batch, seq_len) → Linear → TransformerEncoder → Flatten → FC → latent

    Decoder:
        latent → FC_expand → (batch, seq_len, dim_feedforward)
               → TransformerDecoder → Linear → (batch, seq_len)
    """

    def __init__(
        self,
        input_dim: int,
        latent_dim: int,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 128,
        dropout: float = 0.1
    ):
        super().__init__()

        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.dim_feedforward = dim_feedforward

        # === Input embedding ===
        self.input_embedding = nn.Linear(1, dim_feedforward)
        self.positional_encoding = PositionalEncoding(dim_feedforward, max_len=input_dim)

        # === Encoder transformer ===
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim_feedforward,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # === Latent bottleneck ===
        self.encoder_fc = nn.Linear(dim_feedforward * input_dim, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, dim_feedforward * input_dim)

        # === Decoder transformer ===
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=dim_feedforward,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # Output projection
        self.output_layer = nn.Linear(dim_feedforward, 1)

    def encode(self, x):
        batch = x.size(0)

        x = x.unsqueeze(-1)  # → (batch, seq_len, 1)
        x = self.input_embedding(x)  # → (batch, seq_len, dim_feedforward)

        x = self.positional_encoding(x)

        enc = self.transformer_encoder(x)  # → (batch, seq_len, dim_feedforward)

        flat = enc.reshape(batch, -1)  # → (batch, seq_len * dim_feedforward)
        latent = self.encoder_fc(flat)  # → (batch, latent_dim)
        return latent

    def decode(self, z):
        batch = z.size(0)

        x = self.decoder_fc(z)  # → (batch, seq_len*dim_feedforward)
        x = x.reshape(batch, self.input_dim, self.dim_feedforward)

        # Using self-memory for now (auto-regressive dec not needed)
        dec = self.transformer_decoder(x, x)

        out = self.output_layer(dec).squeeze(-1)  # → (batch, seq_len)
        return out

    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z)
        return recon, z


In [ ]:
class TimeseriesDataset(Dataset):
    """
    Converts a 2D array of shape (n_timeseries, seq_len)
    into a dataset yielding 1 sequence per item.

    Used twice:
      - For AE1: sequences are retention-time column slices (length = M)
      - For AE2: sequences are rows of latent matrices (length = N)
    """

    def __init__(self, timeseries: np.ndarray):
        """
        Args:
            timeseries : np.ndarray with shape (num_seq, seq_len)
        """
        self.timeseries = torch.FloatTensor(timeseries)

    def __len__(self):
        return len(self.timeseries)

    def __getitem__(self, idx):
        """
        Returns:
            A single 1D sequence: shape (seq_len,)
        """
        return self.timeseries[idx]


In [ ]:
def undersample_flat_timeseries(timeseries: np.ndarray, prob: float = 0.25) -> np.ndarray:
    """
    Removes a portion of "flat" timeseries (low variance),
    because GC-IMS matrices contain many empty/noisy columns.

    Args:
        timeseries : shape (num_seq, seq_len)
        prob : probability of *keeping* low-variance sequences
    """
    stds = np.std(timeseries, axis=1)
    median_std = np.median(stds)

    mask = np.ones(len(timeseries), dtype=bool)

    below = stds < median_std
    n_below = np.sum(below)

    # number to keep
    keep_n = int(n_below * prob)

    # which ones to drop
    drop_indices = np.random.choice(
        np.where(below)[0],
        size=n_below - keep_n,
        replace=False
    )

    mask[drop_indices] = False

    print(
        f"Undersampling: kept {mask.sum()}/{len(timeseries)} "
        f"({len(timeseries) - mask.sum()} removed)"
    )

    return timeseries[mask]


In [ ]:
def train_autoencoder(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int,
    learning_rate: float,
    patience: int = 15,
    model_name: str = "autoencoder"
) -> Dict:

    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    history = {'train_loss': [], 'val_loss': []}
    best_val = float("inf")
    wait = 0

    for epoch in range(num_epochs):
        # ------------ Train ------------
        model.train()
        train_losses = []

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()
            recon, _ = model(batch)
            loss = criterion(recon, batch)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        # ------------ Validate ------------
        model.eval()
        val_losses = []

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                recon, _ = model(batch)
                loss = criterion(recon, batch)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        scheduler.step(val_loss)

        # ------------ Early stopping ------------
        if val_loss < best_val:
            best_val = val_loss
            wait = 0
            torch.save(model.state_dict(), f"{config.MODEL_PATH}/{model_name}_best.pt")
        else:
            wait += 1

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{num_epochs} — train={train_loss:.6f}, val={val_loss:.6f}")

    # Load best weights
    model.load_state_dict(torch.load(f"{config.MODEL_PATH}/{model_name}_best.pt"))

    return history


In [ ]:
def extract_columns_as_timeseries(X: np.ndarray) -> np.ndarray:
    """
    X shape: (n_samples, M, N)
    Returns:
        timeseries: (n_samples * N, M)
    """
    n_samples, M, N = X.shape
    # Take each column (RT slice)
    timeseries = X.transpose(0, 2, 1).reshape(-1, M)
    return timeseries


In [ ]:
def extract_rows_as_timeseries(Z: np.ndarray) -> np.ndarray:
    """
    Z shape: (n_samples, D, N)
    Returns:
        timeseries: (n_samples * D, N)
    """
    n_samples, D, N = Z.shape
    timeseries = Z.reshape(-1, N)
    return timeseries


In [ ]:
def encode_dataset_first_autoencoder(model: nn.Module, X: np.ndarray) -> np.ndarray:
    """
    Encodes each column of X using AE1.

    Args:
        X : (n_samples, M, N)
    Returns:
        Z : (n_samples, D, N)
    """
    model.eval()
    n_samples, M, N = X.shape
    Z = np.zeros((n_samples, config.D, N), dtype=np.float32)

    with torch.no_grad():
        for i in tqdm(range(n_samples), desc="Encoding (AE1)"):
            for j in range(N):
                col = torch.tensor(X[i, :, j], dtype=torch.float32).unsqueeze(0).to(device)
                latent = model.encode(col).cpu().numpy()
                Z[i, :, j] = latent

    return Z

In [ ]:
def encode_dataset_second_autoencoder(model: nn.Module, Z: np.ndarray) -> np.ndarray:
    """
    Encodes each latent row using AE2.

    Args:
        Z : (n_samples, D, N)
    Returns:
        E : (n_samples, D, D)
    """
    model.eval()
    n_samples, D, N = Z.shape
    E = np.zeros((n_samples, D, D), dtype=np.float32)

    with torch.no_grad():
        for i in tqdm(range(n_samples), desc="Encoding (AE2)"):
            for j in range(D):
                row = torch.tensor(Z[i, j, :], dtype=torch.float32).unsqueeze(0).to(device)
                latent = model.encode(row).cpu().numpy()
                E[i, j, :] = latent

    return E


In [ ]:
def decode_latent_matrices(model1: nn.Module, model2: nn.Module, E: np.ndarray) -> np.ndarray:
    """
    Decode samples from the fully latent space E → Z → X.

    Args:
        model1 : First autoencoder  (columns / retention time)
        model2 : Second autoencoder (rows / drift evolution)
        E      : (n_samples, D, D)

    Returns:
        X_reconstructed : (n_samples, M, N)
    """
    model1.eval()
    model2.eval()

    n_samples, D, _ = E.shape
    M, N = config.M, config.N

    X_rec = np.zeros((n_samples, M, N), dtype=np.float32)

    with torch.no_grad():
        for i in tqdm(range(n_samples), desc="Decoding latent matrices"):

            # Step 1 — Decode each row via AE2 → reconstruct Z
            Z_rec = np.zeros((D, N), dtype=np.float32)
            for j in range(D):
                row_latent = torch.tensor(E[i, j, :], dtype=torch.float32).unsqueeze(0).to(device)
                row_rec = model2.decode(row_latent).cpu().numpy()
                Z_rec[j, :] = row_rec

            # Step 2 — Decode each column via AE1 → reconstruct X
            for j in range(N):
                col_latent = torch.tensor(Z_rec[:, j], dtype=torch.float32).unsqueeze(0).to(device)
                col_rec = model1.decode(col_latent).cpu().numpy()
                X_rec[i, :, j] = col_rec

    return X_rec



In [ ]:
def generate_synthetic_latent_matrices(E: np.ndarray, y: np.ndarray, label_encoder: LabelEncoder):
    """
    For each class:
        - Flatten all latent matrices
        - Compute mean + covariance
        - Sample same number of synthetic latent samples
        - Reshape into (D, D)

    Returns:
        E_synth : (n_total_synth, D, D)
        y_synth : labels
    """
    unique_labels = np.unique(y)
    E_synth_list = []
    y_synth_list = []

    for label in unique_labels:
        mask = y == label
        E_label = E[mask]
        n_label = len(E_label)

        print(f"Generating {n_label} synthetic samples for class '{label_encoder.inverse_transform([label])[0]}'")

        # Flatten (n_label, D*D)
        flat = E_label.reshape(n_label, -1)

        mean = np.mean(flat, axis=0)
        cov = np.cov(flat.T)
        cov += np.eye(cov.shape[0]) * 1e-6   # regularization

        # Sample
        flat_synth = np.random.multivariate_normal(mean, cov, size=n_label)
        E_synth = flat_synth.reshape(n_label, config.D, config.D)

        E_synth_list.append(E_synth)
        y_synth_list.extend([label] * n_label)

    return np.concatenate(E_synth_list, axis=0), np.array(y_synth_list)


In [ ]:
def plot_training_history(history: Dict, title: str):
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label="Train Loss")
    plt.plot(history['val_loss'], label="Val Loss")
    plt.yscale("log")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    plt.savefig(f"{config.RESULTS_PATH}/{title.replace(' ', '_')}.png", dpi=300)
    plt.show()


In [ ]:
def plot_reconstruction_comparison(original: np.ndarray, reconstructed: np.ndarray, n_samples: int = 3):
    plt.figure(figsize=(12, 4 * n_samples))

    for i in range(n_samples):
        # Original
        plt.subplot(n_samples, 2, 2*i + 1)
        plt.imshow(original[i], aspect='auto', cmap='viridis')
        plt.title(f"Original Sample {i+1}")
        plt.colorbar()

        # Reconstruction
        plt.subplot(n_samples, 2, 2*i + 2)
        plt.imshow(reconstructed[i], aspect='auto', cmap='viridis')
        plt.title(f"Reconstructed Sample {i+1}")
        plt.colorbar()

    plt.tight_layout()
    plt.savefig(f"{config.RESULTS_PATH}/reconstruction_comparison.png", dpi=300)
    plt.show()


In [ ]:
def plot_synthetic_samples(X_synth: np.ndarray, y_synth: np.ndarray, label_encoder: LabelEncoder, n_samples: int = 6):
    unique_labels = np.unique(y_synth)

    plt.figure(figsize=(12, 4 * len(unique_labels)))

    for i, label in enumerate(unique_labels):
        class_samples = X_synth[y_synth == label]
        name = label_encoder.inverse_transform([label])[0]

        for j in range(min(2, len(class_samples))):
            plt.subplot(len(unique_labels), 2, 2*i + j + 1)
            plt.imshow(class_samples[j], aspect='auto', cmap='viridis')
            plt.title(f"Synth {name} — Sample {j+1}")
            plt.colorbar()

    plt.tight_layout()
    plt.savefig(f"{config.RESULTS_PATH}/synthetic_samples.png", dpi=300)
    plt.show()


In [ ]:
def classify_spectra(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
    label_encoder: LabelEncoder,
    n_components: int = 22
):
    """
    Baseline classifier to evaluate augmentation.
    Applies PCA → Random Forest.
    """

    # Flatten (M, N) → (M*N)
    X_train_flat = X_train.reshape(len(X_train), -1)
    X_test_flat  = X_test.reshape(len(X_test),  -1)

    # PCA
    print("\nApplying PCA...")
    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train_flat)
    X_test_pca  = pca.transform(X_test_flat)

    print("Explained variance:", np.sum(pca.explained_variance_ratio_))

    # Random Forest
    print("Training Random Forest classifier...")
    clf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
    clf.fit(X_train_pca, y_train)

    y_pred = clf.predict(X_test_pca)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    print("\nAccuracy:", acc)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cmap='Blues')
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(f"{config.RESULTS_PATH}/confusion_matrix.png", dpi=300)
    plt.show()

    return {
        "accuracy": acc,
        "predictions": y_pred,
        "confusion_matrix": cm,
        "pca": pca,
        "classifier": clf
    }


In [ ]:
def main():
    X_train_full, X_val_full, y_train_full, y_val_full = train_test_split(
        X_processed, y_processed,
        test_size=1 - config.TRAIN_SPLIT,
        random_state=SEED,
        stratify=y_processed
    )

    print("Training samples:", len(X_train_full))
    print("Validation samples:", len(X_val_full))

    print("\nApplying RT pooling to reduce sequence length for AE1...")
    X_train_full = pool_rt_axis(X_train_full, target_M=512)
    X_val_full   = pool_rt_axis(X_val_full,   target_M=512)

    print(f"Transformer AE1 will use input_dim={config.M}")

    print("\nTraining first autoencoder (column encoder)...")
    train_cols = extract_columns_as_timeseries(X_train_full)
    val_cols   = extract_columns_as_timeseries(X_val_full)
    train_cols = undersample_flat_timeseries(train_cols, prob=config.UNDERSAMPLE_PROB)

    ds_train_1 = TimeseriesDataset(train_cols)
    ds_val_1   = TimeseriesDataset(val_cols)

    loader_train_1 = DataLoader(ds_train_1, batch_size=config.BATCH_SIZE, shuffle=True)
    loader_val_1   = DataLoader(ds_val_1,   batch_size=config.BATCH_SIZE, shuffle=False)

    AE1 = TransformerAutoencoder(
        input_dim=config.M,
        latent_dim=config.D,
        nhead=config.NHEAD,
        num_layers=config.NUM_LAYERS,
        dim_feedforward=config.DIM_FEEDFORWARD,
        dropout=config.DROPOUT
    )

    hist1 = train_autoencoder(
        AE1, loader_train_1, loader_val_1,
        num_epochs=config.NUM_EPOCHS,
        learning_rate=config.LEARNING_RATE,
        patience=config.PATIENCE,
        model_name="autoencoder_1"
    )

    plot_training_history(hist1, "AE1 Training Curve")

    print("\nEncoding dataset with AE1...")
    Z_train = encode_dataset_first_autoencoder(AE1, X_train_full)
    Z_val   = encode_dataset_first_autoencoder(AE1, X_val_full)

    print("\nTraining second autoencoder (latent row encoder)...")
    train_rows = extract_rows_as_timeseries(Z_train)
    val_rows   = extract_rows_as_timeseries(Z_val)

    ds_train_2 = TimeseriesDataset(train_rows)
    ds_val_2   = TimeseriesDataset(val_rows)

    loader_train_2 = DataLoader(ds_train_2, batch_size=config.BATCH_SIZE, shuffle=True)
    loader_val_2   = DataLoader(ds_val_2,   batch_size=config.BATCH_SIZE, shuffle=False)

    AE2 = TransformerAutoencoder(
        input_dim=config.N,
        latent_dim=config.D,
        nhead=config.NHEAD,
        num_layers=config.NUM_LAYERS,
        dim_feedforward=config.DIM_FEEDFORWARD,
        dropout=config.DROPOUT
    )

    hist2 = train_autoencoder(
        AE2, loader_train_2, loader_val_2,
        num_epochs=config.NUM_EPOCHS,
        learning_rate=config.LEARNING_RATE,
        patience=config.PATIENCE,
        model_name="autoencoder_2"
    )

    plot_training_history(hist2, "AE2 Training Curve")

    print("\nEncoding dataset with AE2...")
    E_train = encode_dataset_second_autoencoder(AE2, Z_train)
    E_val   = encode_dataset_second_autoencoder(AE2, Z_val)

    print("\nReconstructing validation samples...")
    X_rec = decode_latent_matrices(AE1, AE2, E_val[:3])
    X_rec = preprocessor.inverse_transform(X_rec)
    X_val_original = preprocessor.inverse_transform(X_val_full[:3])
    plot_reconstruction_comparison(X_val_original, X_rec)

    print("\nGenerating synthetic latent matrices...")
    E_synth, y_synth = generate_synthetic_latent_matrices(E_train, y_train_full, label_encoder)

    print("Decoding synthetic matrices to full spectra...")
    X_synth = decode_latent_matrices(AE1, AE2, E_synth)
    X_synth = preprocessor.inverse_transform(X_synth)
    plot_synthetic_samples(X_synth, y_synth, label_encoder)

    print("\nRunning baseline classification...")
    results_baseline = classify_spectra(
        preprocessor.inverse_transform(X_train_full),
        y_train_full,
        preprocessor.inverse_transform(X_val_full),
        y_val_full,
        label_encoder
    )

    print("\nRunning augmented classification...")
    X_train_aug = np.concatenate([preprocessor.inverse_transform(X_train_full), X_synth], axis=0)
    y_train_aug = np.concatenate([y_train_full, y_synth])

    results_aug = classify_spectra(
        X_train_aug, y_train_aug,
        preprocessor.inverse_transform(X_val_full),
        y_val_full,
        label_encoder
    )

    print("\nPipeline Completed Successfully!")

    return {
        "AE1": AE1,
        "AE2": AE2,
        "E_train": E_train,
        "E_val": E_val,
        "X_synth": X_synth,
        "y_synth": y_synth,
        "baseline": results_baseline,
        "augmented": results_aug,
    }


In [ ]:
pipeline_results = main()

In [ ]:
def load_trained_pipeline(model_path: str = None):
    if model_path is None:
        model_path = f"{config.MODEL_PATH}/complete_pipeline.pt"

    checkpoint = torch.load(model_path, map_location=device)

    AE1 = TransformerAutoencoder(config.M, config.D)
    AE1.load_state_dict(checkpoint['autoencoder_1'])

    AE2 = TransformerAutoencoder(config.N, config.D)
    AE2.load_state_dict(checkpoint['autoencoder_2'])

    preprocessor = SpectralDataPreprocessor()
    preprocessor.log_min = checkpoint['preprocessor_log_min']
    preprocessor.log_max = checkpoint['preprocessor_log_max']

    label_encoder = checkpoint['label_encoder']

    return {
        "AE1": AE1,
        "AE2": AE2,
        "preprocessor": preprocessor,
        "label_encoder": label_encoder
    }


In [ ]:
def generate_new_synthetic_samples(
    n_per_class: int,
    AE1, AE2,
    E_train, y_train,
    label_encoder,
    preprocessor
):
    unique_labels = np.unique(y_train)

    E_synth_list = []
    y_synth_list = []

    for label in unique_labels:
        E_label = E_train[y_train == label]
        flat = E_label.reshape(len(E_label), -1)

        mean = flat.mean(axis=0)
        cov = np.cov(flat.T) + np.eye(flat.shape[1]) * 1e-6

        flat_synth = np.random.multivariate_normal(mean, cov, size=n_per_class)
        E_synth = flat_synth.reshape(n_per_class, config.D, config.D)

        E_synth_list.append(E_synth)
        y_synth_list.extend([label] * n_per_class)

    E_synth_total = np.concatenate(E_synth_list)
    y_synth_total = np.array(y_synth_list)

    X_synth = decode_latent_matrices(AE1, AE2, E_synth_total)
    X_synth = preprocessor.inverse_transform(X_synth)

    return X_synth, y_synth_total
